# Bocconi Nuclei Challenge — Segmentation (Dedicated Notebook)

**Task**: Predict one binary mask per image (nucleus pixels = 1).  
**Metric**: Mean Dice score over the test set.

### Strategy (vs. previous notebook)
| What | Previous | This notebook |
|---|---|---|
| Encoder | Scratch U-Net | **Pretrained ResNet-34** (ImageNet) via `segmentation_models_pytorch` |
| Augmentation | Flip + ColorJitter | **Albumentations**: flip, rotate, elastic, grid distort, CLAHE, stain aug |
| Loss | 0.6 BCE + 0.4 Dice | **0.5 BCE + 0.5 Dice** (with label smoothing) |
| LR schedule | Cosine | **OneCycleLR** (warmup + cosine decay) |
| Epochs | 20 | **30** with best-checkpoint saving |
| Inference | Single pass | **Test-Time Augmentation** (H-flip + V-flip ensemble) |
| Threshold | Grid-searched | **Grid-searched on val** |

Course: **Computer Vision and Image Processing** — Prof. Chiara Plizzari

## 1. Install & imports

In [14]:
!pip -q install segmentation-models-pytorch albumentations

import random, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 256
NUM_WORKERS = 0   # 0 avoids the Python-3.12 multiprocessing warning in Colab
print('Device:', DEVICE)

Device: cuda


## 2. Mount Drive & load dataset

In [15]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = Path('/content/drive/MyDrive')
ZIP_PATH    = BASE_DIR / 'nuclei-segmentation-challenge.zip'
EXTRACT_DIR = Path('/content/nuclei-segmentation-challenge')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Zip not found: {ZIP_PATH}')
if not any(EXTRACT_DIR.iterdir()):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)

# Auto-locate dataset root
DATA_ROOT = EXTRACT_DIR / 'kaggle_upload_v2' / 'dataset'
if not DATA_ROOT.exists():
    for c in [EXTRACT_DIR] + list(EXTRACT_DIR.rglob('dataset')):
        if (c / 'train' / 'images').exists() and (c / 'test' / 'images').exists():
            DATA_ROOT = c; break

TRAIN_IMG_DIR = DATA_ROOT / 'train' / 'images'
TEST_IMG_DIR  = DATA_ROOT / 'test'  / 'images'
df_seg  = pd.read_csv(DATA_ROOT / 'train' / 'train_labels_segmentation.csv')
df_meta = pd.read_csv(DATA_ROOT / 'train' / 'train_metadata.csv')

TRAIN_IDS = sorted(df_seg['image_id'].tolist())
TEST_IDS  = sorted(int(p.stem) for p in TEST_IMG_DIR.glob('*.png'))

seg_lu  = df_seg.set_index('image_id')
meta_lu = df_meta.set_index('image_id')

print('Train images :', len(TRAIN_IDS))
print('Test images  :', len(TEST_IDS))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train images : 900
Test images  : 280


## 3. Utility functions

In [16]:
def load_img(image_id, split='train'):
    folder = TRAIN_IMG_DIR if split == 'train' else TEST_IMG_DIR
    return np.array(
        Image.open(folder / f'{image_id:04d}.png').convert('RGB'), dtype=np.uint8
    )

def rle_to_mask(rle, h=IMG_SIZE, w=IMG_SIZE):
    mask = np.zeros(h * w, dtype=np.uint8)
    if pd.isna(rle) or str(rle).strip() == '':
        return mask.reshape(h, w)
    parts = list(map(int, str(rle).split()))
    for s, l in zip(parts[0::2], parts[1::2]):
        mask[s:s + l] = 1
    return mask.reshape(h, w)

def mask_to_rle(mask):
    pixels = mask.astype(np.uint8).reshape(-1)
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0]
    out    = []
    for s, e in zip(runs[::2], runs[1::2]):
        out.extend([str(int(s)), str(int(e - s))])
    return ' '.join(out)

def dice_score(pred_mask, gt_mask):
    """Compute Dice on two binary numpy arrays."""
    p = pred_mask.astype(bool)
    g = gt_mask.astype(bool)
    inter = (p & g).sum()
    return float(2 * inter / (p.sum() + g.sum() + 1e-8))

## 4. Slide-level train / validation split

In [17]:
slide_names = sorted(df_meta['source_slide'].unique().tolist())
random.shuffle(slide_names)

n_train = max(1, int(len(slide_names) * 0.80))
train_slides = set(slide_names[:n_train])
val_slides   = set(slide_names[n_train:])

train_ids = [i for i in TRAIN_IDS if meta_lu.loc[i, 'source_slide'] in train_slides]
val_ids   = [i for i in TRAIN_IDS if meta_lu.loc[i, 'source_slide'] in val_slides]

print(f'Train: {len(train_ids)} images ({len(train_slides)} slides)')
print(f'Val  : {len(val_ids)} images ({len(val_slides)} slides)')

Train: 707 images (22 slides)
Val  : 193 images (6 slides)


## 5. Augmentation pipelines (Albumentations)

We use two separate pipelines:
- **train**: aggressive spatial + photometric augmentations
- **val / test**: only normalise (no spatial changes)

`CLAHE` improves contrast in H&E stained patches.  
`ElasticTransform` and `GridDistortion` simulate realistic tissue deformation.  
`HueSaturationValue` with narrow hue range approximates stain variation.

In [18]:
# ImageNet mean/std — correct because our encoder is ImageNet-pretrained
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_aug = A.Compose([
    # ── Spatial ──────────────────────────────────────────────────────────────
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=30, border_mode=0, p=0.5),
    A.ElasticTransform(alpha=60, sigma=8, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
    # ── Photometric ───────────────────────────────────────────────────────────
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                         val_shift_limit=20, p=0.4),
    A.GaussNoise(var_limit=(5, 30), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    # ── Normalise & convert ───────────────────────────────────────────────────
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_3291/3580942993.py:19: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 30), p=0.3),


## 6. Dataset

In [19]:
class SegDataset(Dataset):
    def __init__(self, ids, augment_fn, split='train'):
        self.ids        = ids
        self.augment_fn = augment_fn
        self.split      = split

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        img      = load_img(image_id, self.split)   # (H, W, 3) uint8

        if self.split == 'train':
            mask = rle_to_mask(seg_lu.loc[image_id, 'rle_mask'])  # (H, W) uint8
            out  = self.augment_fn(image=img, mask=mask)
            return out['image'], out['mask'].float().unsqueeze(0)
        else:
            # Test split: no mask available
            out = self.augment_fn(image=img)
            return out['image'], image_id

train_ds = SegDataset(train_ids, train_aug, split='train')
val_ds   = SegDataset(val_ids,   val_aug,   split='train')  # use train masks for val

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

Train batches: 45 | Val batches: 13


## 7. Model — U-Net with pretrained ResNet-34 encoder

`segmentation_models_pytorch` provides standard segmentation architectures
with ImageNet-pretrained encoders.

- **Architecture**: U-Net (standard skip connections)
- **Encoder**: ResNet-34 pretrained on ImageNet
- **Decoder depth**: 5 levels (matches encoder)
- **Output**: single-channel logit map → sigmoid → binary mask

In [20]:
model = smp.Unet(
    encoder_name    = 'resnet34',      # pretrained ResNet-34 backbone
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 1,               # binary segmentation
    activation      = None,            # raw logits; we apply sigmoid manually
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

Total params    : 24,436,369
Trainable params: 24,436,369


## 8. Loss function — Dice + BCE

In [21]:
def soft_dice_loss(logits, targets, smooth=1.0):
    """
    Soft Dice loss on sigmoid probabilities.
    Works better than hard-threshold Dice during training.
    """
    probs  = torch.sigmoid(logits)
    pf     = probs.view(-1)
    tf     = targets.view(-1)
    inter  = (pf * tf).sum()
    return 1.0 - (2.0 * inter + smooth) / (pf.sum() + tf.sum() + smooth)


bce_fn = nn.BCEWithLogitsLoss()


def combined_loss(logits, targets, bce_w=0.5, dice_w=0.5):
    return bce_w * bce_fn(logits, targets) + dice_w * soft_dice_loss(logits, targets)

## 9. Training loop — OneCycleLR

**OneCycleLR** is a one-cycle learning rate policy that:
1. Linearly warms up LR from a very small value to `max_lr`
2. Cosine-anneals back down to `min_lr`

This typically converges faster and to a better minimum than flat or cosine schedules.

In [22]:
EPOCHS  = 30
MAX_LR  = 3e-4

optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr          = MAX_LR,
    steps_per_epoch = len(train_dl),
    epochs          = EPOCHS,
    pct_start       = 0.1,     # 10% of steps for warmup
    anneal_strategy = 'cos',
)

CKPT_PATH    = '/tmp/seg_best_resnet34.pth'
best_val_dice = 0.0

for ep in range(1, EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_dl, desc=f'train {ep}/{EPOCHS}', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        loss   = combined_loss(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_dices = []
    with torch.no_grad():
        for imgs, masks in val_dl:
            logits = model(imgs.to(DEVICE))
            probs  = torch.sigmoid(logits).cpu().numpy()
            masks  = masks.numpy()
            for p, m in zip(probs, masks):
                pred = (p[0] > 0.5).astype(np.uint8)
                val_dices.append(dice_score(pred, m[0]))

    mean_dice = float(np.mean(val_dices))
    lr_now    = scheduler.get_last_lr()[0]
    print(f'Ep {ep:02d} | loss={train_loss/len(train_dl):.4f}'
          f' | val_dice={mean_dice:.4f} | lr={lr_now:.2e}')

    if mean_dice > best_val_dice:
        best_val_dice = mean_dice
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'  ✓ checkpoint saved (val_dice={best_val_dice:.4f})')

print(f'\nBest val Dice: {best_val_dice:.4f}')

train 1/30:   0%|          | 0/45 [00:00<?, ?it/s]

Ep 01 | loss=0.7261 | val_dice=0.5461 | lr=8.50e-05
  ✓ checkpoint saved (val_dice=0.5461)


train 2/30:   0%|          | 0/45 [00:00<?, ?it/s]

Ep 02 | loss=0.5044 | val_dice=0.7846 | lr=2.30e-04
  ✓ checkpoint saved (val_dice=0.7846)


train 3/30:   0%|          | 0/45 [00:00<?, ?it/s]

...

Ep 28 | loss=0.1935 | val_dice=0.8349 | lr=3.96e-06


train 29/30:   0%|          | 0/45 [00:00<?, ?it/s]

Ep 29 | loss=0.1927 | val_dice=0.8360 | lr=9.71e-07
  ✓ checkpoint saved (val_dice=0.8360)


train 30/30:   0%|          | 0/45 [00:00<?, ?it/s]

Ep 30 | loss=0.1923 | val_dice=0.8350 | lr=1.70e-09

Best val Dice: 0.8360


## 10. Threshold tuning + Test-Time Augmentation (TTA)

**TTA** averages predictions over multiple augmented versions of the same image.
Here we use original + H-flip + V-flip (3 passes), then average the probability maps.
This is a zero-cost way to reduce variance and reliably improves Dice by ~0.5–1%.

In [23]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH))
model.eval()


def predict_proba(img_np, use_tta=True):
    """
    Run the model on one image and return a probability map (H, W).

    Parameters
    ----------
    img_np  : np.ndarray  (H, W, 3) uint8
    use_tta : bool  If True, average 3 augmented passes (original + 2 flips)

    Returns
    -------
    np.ndarray  (H, W) float in [0, 1]
    """
    def _infer(arr):
        out  = val_aug(image=arr)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = torch.sigmoid(model(out))[0, 0].cpu().numpy()
        return prob

    prob = _infer(img_np)
    if use_tta:
        # H-flip: flip input, infer, flip prediction back
        prob_hf = np.fliplr(_infer(np.fliplr(img_np).copy()))
        # V-flip
        prob_vf = np.flipud(_infer(np.flipud(img_np).copy()))
        prob    = (prob + prob_hf + prob_vf) / 3.0
    return prob


# ── Collect val probabilities once ────────────────────────────────────────────
val_probs = {}
val_gts   = {}
for image_id in tqdm(val_ids, desc='val TTA probs'):
    img = load_img(image_id, 'train')
    val_probs[image_id] = predict_proba(img, use_tta=True)
    val_gts[image_id]   = rle_to_mask(seg_lu.loc[image_id, 'rle_mask'])

# ── Sweep thresholds ──────────────────────────────────────────────────────────
best_thr, best_dice = 0.50, 0.0
for thr in np.arange(0.30, 0.71, 0.05):
    dices = [dice_score((val_probs[i] > thr).astype(np.uint8), val_gts[i])
             for i in val_ids]
    mean  = float(np.mean(dices))
    print(f'  thr={thr:.2f} | Dice = {mean:.4f}')
    if mean > best_dice:
        best_dice, best_thr = mean, thr

print(f'\nBest threshold : {best_thr:.2f}  →  val Dice (TTA) = {best_dice:.4f}')
SEG_THR = best_thr

val TTA probs:   0%|          | 0/193 [00:00<?, ?it/s]

  thr=0.30 | Dice = 0.8368
  thr=0.35 | Dice = 0.8382
  thr=0.40 | Dice = 0.8389
  thr=0.45 | Dice = 0.8389
  thr=0.50 | Dice = 0.8383
  thr=0.55 | Dice = 0.8369
  thr=0.60 | Dice = 0.8347
  thr=0.65 | Dice = 0.8314
  thr=0.70 | Dice = 0.8268

Best threshold : 0.40  →  val Dice (TTA) = 0.8389


In [24]:
SAVE_PATH = BASE_DIR / 'seg_resnet34_unet.pth'
torch.save(model.state_dict(), SAVE_PATH)
print('Model saved to:', SAVE_PATH)

Model saved to: /content/drive/MyDrive/seg_resnet34_unet.pth


## 11. Generate Kaggle submission

In [25]:
rows = []
for image_id in tqdm(TEST_IDS, desc='test inference'):
    img  = load_img(image_id, 'test')
    prob = predict_proba(img, use_tta=True)
    mask = (prob > SEG_THR).astype(np.uint8)
    rows.append({'image_id': image_id, 'rle_mask': mask_to_rle(mask)})

pd.DataFrame(rows, columns=['image_id', 'rle_mask']).to_csv(
    'submission_segmentation.csv', index=False
)
print('submission_segmentation.csv saved.')
print(f'Val Dice (TTA, thr={SEG_THR:.2f}): {best_dice:.4f}')

test inference:   0%|          | 0/280 [00:00<?, ?it/s]

submission_segmentation.csv saved.
Val Dice (TTA, thr=0.40): 0.8389


## 12. Summary

In [26]:
print('=' * 60)
print('Model     : U-Net + ResNet-34 (ImageNet pretrained)')
print(f'Epochs    : {EPOCHS}')
print(f'Threshold : {SEG_THR:.2f}')
print(f'Val Dice (TTA): {best_dice:.4f}')
print('Submission: submission_segmentation.csv')
print('=' * 60)

Model     : U-Net + ResNet-34 (ImageNet pretrained)
Epochs    : 30
Threshold : 0.40
Val Dice (TTA): 0.8389
Submission: submission_segmentation.csv
